# Generate Mock Point-Cloud LandscapeFiles

This notebook writes synthetic Celldega `LandscapeFiles` for large point-cloud performance testing. It intentionally writes no transcript tiles. Each generated dataset contains cell positions, cluster assignments, 50 sparse mock gene-expression files, and the metadata Celldega needs to color cells by cluster or gene.

Default sizes are 0.5M, 1M, and 2M cells. Add more values to `CELL_COUNTS` if you want larger stress tests.

In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [2]:
from __future__ import annotations

import colorsys
import json
import math
import shutil
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq


OUTPUT_ROOT = Path("data/mock_point_cloud_landscapes").resolve()
CELL_COUNTS = [500_000, 1_000_000, 2_000_000, 5_000_000, 10_000_000]
GENES = [f"MockGene{i:02d}" for i in range(1, 51)]
N_CLUSTERS = 12
EXPRESSION_FRACTION_RANGE = (0.01, 0.15)
CHUNK_SIZE = 250_000
SEED = 42
OVERWRITE = False

OUTPUT_ROOT

PosixPath('/Users/feni/Documents/celldega/notebooks/data/mock_point_cloud_landscapes')

In [3]:
def make_palette(n: int, saturation: float = 0.68, value: float = 0.88) -> list[str]:
    """Generate visually distinct hex colors without requiring matplotlib."""
    colors = []
    for i in range(n):
        hue = (i / max(n, 1) + 0.03) % 1.0
        r, g, b = colorsys.hsv_to_rgb(hue, saturation, value)
        colors.append(f"#{int(r * 255):02x}{int(g * 255):02x}{int(b * 255):02x}")
    return colors


def cell_names(start: int, stop: int) -> list[str]:
    return [f"cell_{i:09d}" for i in range(start, stop)]


def make_coordinates(rng: np.random.Generator, n: int) -> np.ndarray:
    """Return an n x 3 float32 point cloud with a mildly tissue-like footprint."""
    coords = np.empty((n, 3), dtype=np.float32)

    # A large flat-ish spatial field plus a little 3D thickness.
    coords[:, 0] = rng.uniform(0, 5_000, size=n)
    coords[:, 1] = rng.uniform(0, 5_000, size=n)
    coords[:, 2] = rng.normal(loc=0, scale=600, size=n)

    # Add gentle global curvature so rotation has visible depth structure.
    x_centered = (coords[:, 0] - 10_000) / 10_000
    y_centered = (coords[:, 1] - 10_000) / 10_000
    coords[:, 2] += (450 * np.sin(x_centered * math.pi) * np.cos(y_centered * math.pi)).astype(
        np.float32
    )
    return coords


def write_cell_metadata(out_dir: Path, n_cells: int, rng: np.random.Generator, chunk_size: int) -> None:
    path = out_dir / "cell_metadata.parquet"
    writer = None
    try:
        for start in range(0, n_cells, chunk_size):
            stop = min(start + chunk_size, n_cells)
            coords = make_coordinates(rng, stop - start)
            geometry = pa.FixedSizeListArray.from_arrays(
                pa.array(coords.reshape(-1), type=pa.float32()),
                list_size=3,
            )
            table = pa.table(
                {
                    "name": pa.array(cell_names(start, stop), type=pa.string()),
                    "geometry": geometry,
                }
            )
            if writer is None:
                writer = pq.ParquetWriter(path, table.schema, compression="snappy")
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()


def write_clusters(
    out_dir: Path,
    n_cells: int,
    rng: np.random.Generator,
    chunk_size: int,
    n_clusters: int,
) -> list[str]:
    cluster_dir = out_dir / "cell_clusters"
    cluster_dir.mkdir(parents=True, exist_ok=True)

    clusters = [f"cluster_{i:02d}" for i in range(1, n_clusters + 1)]
    cluster_array = np.asarray(clusters, dtype=object)
    probabilities = rng.dirichlet(np.ones(n_clusters) * 2.0)
    counts = np.zeros(n_clusters, dtype=np.int64)

    writer = None
    try:
        for start in range(0, n_cells, chunk_size):
            stop = min(start + chunk_size, n_cells)
            cluster_ids = rng.choice(n_clusters, size=stop - start, p=probabilities)
            counts += np.bincount(cluster_ids, minlength=n_clusters)
            table = pa.table(
                {
                    "__index_level_0__": pa.array(cell_names(start, stop), type=pa.string()),
                    "cluster": pa.array(cluster_array[cluster_ids], type=pa.string()),
                }
            )
            if writer is None:
                writer = pq.ParquetWriter(
                    cluster_dir / "cluster.parquet", table.schema, compression="snappy"
                )
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()

    colors = make_palette(n_clusters)
    meta_cluster = pa.table(
        {
            "__index_level_0__": pa.array(clusters, type=pa.string()),
            "color": pa.array(colors, type=pa.string()),
            "count": pa.array(counts, type=pa.int64()),
        }
    )
    pq.write_table(meta_cluster, cluster_dir / "meta_cluster.parquet", compression="snappy")
    return clusters


def write_gene_expression(
    out_dir: Path,
    n_cells: int,
    rng: np.random.Generator,
    genes: list[str],
    expression_fraction_range: tuple[float, float],
) -> None:
    cbg_dir = out_dir / "cbg"
    cbg_dir.mkdir(parents=True, exist_ok=True)

    colors = make_palette(len(genes), saturation=0.72, value=0.82)
    gene_names = []
    means = []
    stds = []
    maxes = []
    nonzeros = []

    low, high = expression_fraction_range
    for gene in genes:
        fraction = float(rng.uniform(low, high))
        n_positive = max(1, int(round(n_cells * fraction)))

        cell_ids = rng.choice(n_cells, size=n_positive, replace=False).astype(np.int32)
        cell_ids.sort()

        # Small integer counts, with a long-ish tail but bounded for compact storage.
        expr = (rng.negative_binomial(n=2, p=0.45, size=n_positive) + 1).astype(np.int16)
        expr = np.clip(expr, 1, 255).astype(np.int16)

        table = pa.table(
            {
                "__index_level_0__": pa.array(cell_ids, type=pa.int32()),
                gene: pa.array(expr, type=pa.int16()),
            }
        )
        pq.write_table(table, cbg_dir / f"{gene}.parquet", compression="snappy")

        sum_expr = float(expr.sum())
        sumsq_expr = float(np.square(expr.astype(np.float64)).sum())
        mean = sum_expr / n_cells
        variance = max(0.0, (sumsq_expr / n_cells) - mean**2)

        gene_names.append(gene)
        means.append(mean)
        stds.append(math.sqrt(variance))
        maxes.append(int(expr.max()))
        nonzeros.append(n_positive / n_cells)

    meta_gene = pa.table(
        {
            "__index_level_0__": pa.array(gene_names, type=pa.string()),
            "mean": pa.array(means, type=pa.float64()),
            "std": pa.array(stds, type=pa.float64()),
            "max": pa.array(maxes, type=pa.int64()),
            "non-zero": pa.array(nonzeros, type=pa.float64()),
            "color": pa.array(colors, type=pa.string()),
        }
    )
    pq.write_table(meta_gene, out_dir / "meta_gene.parquet", compression="snappy")


def write_landscape_parameters(out_dir: Path, n_cells: int, tile_size: int = 1_000) -> None:
    params = {
        "technology": "point-cloud",
        "segmentation_approach": ["default"],
        "max_pyramid_zoom": None,
        "tile_size": tile_size,
        "image_info": [],
        "image_format": ".webp",
        "use_int_index": True,
        "use_row_groups": False,
        "mock_dataset": {
            "n_cells": n_cells,
            "n_genes": len(GENES),
            "expression_fraction_range": list(EXPRESSION_FRACTION_RANGE),
            "note": "No transcript_tiles are written; this is for point-cloud performance testing.",
        },
    }
    with (out_dir / "landscape_parameters.json").open("w") as f:
        json.dump(params, f, indent=2)


def generate_mock_landscape(
    n_cells: int,
    out_dir: Path,
    *,
    seed: int = 0,
    overwrite: bool = False,
    chunk_size: int = CHUNK_SIZE,
    genes: list[str] = GENES,
    expression_fraction_range: tuple[float, float] = EXPRESSION_FRACTION_RANGE,
    n_clusters: int = N_CLUSTERS,
) -> Path:
    out_dir = Path(out_dir)
    if out_dir.exists():
        if not overwrite:
            raise FileExistsError(f"{out_dir} already exists. Set overwrite=True to replace it.")
        shutil.rmtree(out_dir)

    out_dir.mkdir(parents=True)
    (out_dir / "pyramid_images").mkdir(exist_ok=True)

    rng = np.random.default_rng(seed)
    print(f"Writing {n_cells:,} cells to {out_dir}")

    write_cell_metadata(out_dir, n_cells, rng, chunk_size)
    write_clusters(out_dir, n_cells, rng, chunk_size, n_clusters)
    write_gene_expression(out_dir, n_cells, rng, genes, expression_fraction_range)
    write_landscape_parameters(out_dir, n_cells)

    print(f"Done: {out_dir}")
    return out_dir

In [4]:
# # Generate the default performance fixtures.
# # This writes sizeable parquet files; set OVERWRITE=True above when regenerating.
# generated = []
# for i, n_cells in enumerate(CELL_COUNTS):
#     label = f"mock_point_cloud_{n_cells // 1_000}k"
#     generated.append(
#         generate_mock_landscape(
#             n_cells,
#             OUTPUT_ROOT / label,
#             seed=SEED + i,
#             overwrite=OVERWRITE,
#         )
#     )

# generated

In [5]:
import celldega as dega

/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/h5py/__init__.py:36: UserWarning: h5py is running against HDF5 1.14.5 when it was built against 1.14.6, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "
/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/Users/feni/Documents/celldega/dega/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources im

In [6]:
server_address = dega.viz.get_local_server()
base_url = f'http://localhost:{server_address}/data/mock_point_cloud_landscapes/mock_point_cloud_5000k'

In [7]:
base_url

'http://localhost:61502/data/mock_point_cloud_landscapes/mock_point_cloud_5000k'

In [8]:
# Optional: preview one fixture in Celldega.
# Run this from a terminal in OUTPUT_ROOT first:
#   python -m http.server 8010

# Then uncomment this cell:
from celldega import Landscape
Landscape(
    base_url=base_url,
    height=700,
    rotation_x=25,
    technology='point-cloud', 
)

/var/folders/8d/jxpy9rd10j7fp2rcj_s5sz3c0000gq/T/ipykernel_39108/3816124889.py:7: UserWarning: Transformation matrix not found at http://localhost:61502/data/mock_point_cloud_landscapes/mock_point_cloud_5000k/micron_to_image_transform.csv. Using identity.
  Landscape(
